In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cmocean as cmo
import xarray as xr

from pathlib import Path
from oceanum.datamesh import Connector

In [2]:
TOKEN = (Path.home() / ".datamesh_token").read_text().strip()
datamesh = Connector(token=TOKEN)
print("Connected to", datamesh.host)

Connected to datamesh.oceanum.io


In [ ]:
DATASET = "ecmwf_ifs_0p25_oper_sfc"
BBOX = [161, -51, 185, -31]

def run_test(label, times):
    ds = datamesh.query(
        datasource=DATASET,
        variables=["t2m"],
        timefilter={"type": "range", "times": times},
        geofilter={"type": "bbox", "geom": BBOX},
    )
    print(f"\n=== {label} ===")
    print(f"datasource: {DATASET}")
    print(f"bbox:       {BBOX}")
    print(f"requested:  {times[0]} -> {times[1]}")
    if ds is None:
        print("RESULT: NO DATA returned")
    else:
        print(f"RESULT: data returned, {ds.sizes['time']} time steps "
              f"({ds.time.min().values} -> {ds.time.max().values})")
    return ds

# Test 1: control. Current forecast, should return data.
ds1 = run_test("TEST 1: current period (control)",
               ["2026-10-20T06:00", "2026-10-26T12:00"])

# Test 2: requested period. Returns no data.
ds2 = run_test("TEST 2: requested period, Jan 2026",
               ["2026-01-01", "2026-01-02"])

# Test 3: control. Current forecast, should return data.
ds3 = run_test("TEST 3: current period (control)",
               ["2026-10-26T06:00", "2026-10-30T12:00"])

# Test 4: requested period. Returns no data.
ds4 = run_test("TEST 4: requested period, Jan 2026",
               ["2026-01-01", "2026-03-30"])


=== TEST 1: current period (control) ===
datasource: ecmwf_ifs_0p25_oper_sfc
bbox:       [161, -51, 185, -31]
requested:  2026-10-20T06:00 -> 2026-10-26T12:00
RESULT: data returned, 49 time steps (2026-10-20T12:00:00.000000000 -> 2026-10-26T12:00:00.000000000)


/home/nn411/miniforge3/envs/moana_python/lib/python3.14/site-packages/oceanum/datamesh/connection.py:326: UserWarning: No data found for query
  warnings.warn("No data found for query")



=== TEST 2: requested period, Jan 2026 ===
datasource: ecmwf_ifs_0p25_oper_sfc
bbox:       [161, -51, 185, -31]
requested:  2026-01-01 -> 2026-01-02
RESULT: NO DATA returned

=== TEST 3: current period (control) ===
datasource: ecmwf_ifs_0p25_oper_sfc
bbox:       [161, -51, 185, -31]
requested:  2026-10-26T06:00 -> 2026-10-30T12:00
RESULT: data returned, 19 time steps (2026-10-26T06:00:00.000000000 -> 2026-10-30T12:00:00.000000000)

=== TEST 3: requested period, Jan 2026 ===
datasource: ecmwf_ifs_0p25_oper_sfc
bbox:       [161, -51, 185, -31]
requested:  2026-01-01 -> 2026-03-30
RESULT: NO DATA returned


/home/nn411/miniforge3/envs/moana_python/lib/python3.14/site-packages/oceanum/datamesh/connection.py:326: UserWarning: No data found for query
  warnings.warn("No data found for query")


In [7]:
DATASET_2024 = "ecmwf_ifs_0p25_oper_sfc_2024"
BBOX = [161, -51, 185, -31]
needed = ["u10", "v10", "t2m", "d2m", "msl", "tp", "ssr", "strd"]

# Part 1: are the variables listed?
dsrc = datamesh.get_datasource(DATASET_2024)
names = list(dsrc.variables)
print(f"=== {DATASET_2024} ===")
print("catalog time range:", dsrc.tstart, "->", dsrc.tend)
print("missing from catalog:", [v for v in needed if v not in names])

# Part 2: is there data?
periods = {
    "Nov 15 2024 (inside stored run)": ["2024-11-15", "2024-11-16"],
    "Apr 1 2024 (catalog says available)": ["2024-04-01", "2024-04-02"],
}

for label, times in periods.items():
    print(f"\n--- {label}: {times[0]} -> {times[1]} ---")
    for v in needed:
        ds = datamesh.query(
            datasource=DATASET_2024,
            variables=[v],
            timefilter={"type": "range", "times": times},
            geofilter={"type": "bbox", "geom": BBOX},
        )
        if ds is None:
            print(f"{v:5s}: NO DATA")
        else:
            print(f"{v:5s}: data, {ds.sizes['time']} time steps")

=== ecmwf_ifs_0p25_oper_sfc_2024 ===
catalog time range: 2024-03-18 12:00:00+00:00 -> 2024-11-22 00:00:00+00:00
missing from catalog: []

--- Nov 15 2024 (inside stored run): 2024-11-15 -> 2024-11-16 ---
u10  : data, 9 time steps
v10  : data, 9 time steps
t2m  : data, 9 time steps
d2m  : data, 9 time steps
msl  : data, 9 time steps
tp   : data, 9 time steps
ssr  : data, 9 time steps
strd : data, 9 time steps

--- Apr 1 2024 (catalog says available): 2024-04-01 -> 2024-04-02 ---


/home/nn411/miniforge3/envs/moana_python/lib/python3.14/site-packages/oceanum/datamesh/connection.py:326: UserWarning: No data found for query
  warnings.warn("No data found for query")


u10  : NO DATA
v10  : NO DATA
t2m  : NO DATA
d2m  : NO DATA
msl  : NO DATA
tp   : NO DATA
ssr  : NO DATA
strd : NO DATA


In [8]:
DATASET_2024  = "ecmwf_ifs_0p25_oper_sfc_2024"
VARIABLE = "t2m"
START    = "2024-03-18"
END      = "2024-11-22"

for d in pd.date_range(START, END, freq="7D"):
    ds = datamesh.query(
        datasource=DATASET_2024,
        variables=[VARIABLE],
        timefilter={"type": "range", "times": [str(d), str(d + pd.Timedelta("1D"))]},
        geofilter={"type": "bbox", "geom": [161, -51, 185, -31]},
    )
    print(d.date(), "DATA" if ds is not None else "no data")

/home/nn411/miniforge3/envs/moana_python/lib/python3.14/site-packages/oceanum/datamesh/connection.py:326: UserWarning: No data found for query
  warnings.warn("No data found for query")


2024-03-18 no data
2024-03-25 no data
2024-04-01 no data
2024-04-08 no data
2024-04-15 no data
2024-04-22 no data
2024-04-29 no data
2024-05-06 no data
2024-05-13 no data
2024-05-20 no data
2024-05-27 no data
2024-06-03 no data
2024-06-10 no data
2024-06-17 no data
2024-06-24 no data
2024-07-01 no data
2024-07-08 no data
2024-07-15 no data
2024-07-22 no data
2024-07-29 no data
2024-08-05 no data
2024-08-12 no data
2024-08-19 no data
2024-08-26 no data
2024-09-02 no data
2024-09-09 no data
2024-09-16 no data
2024-09-23 no data
2024-09-30 no data
2024-10-07 no data
2024-10-14 no data
2024-10-21 no data
2024-10-28 no data
2024-11-04 no data
2024-11-11 DATA
2024-11-18 DATA


In [13]:
DATASET_era  = "era5_temp2m"
VARIABLE = "t2m"   # check the real name with list(datamesh.get_datasource(DATASET).variables)
START    = "2026-01-01"
END      = "2026-03-31"

for d in pd.date_range(START, END, freq="7D"):
    ds = datamesh.query(
        datasource=DATASET_era,
        variables=[VARIABLE],
        timefilter={"type": "range", "times": [str(d), str(d + pd.Timedelta("1D"))]},
        geofilter={"type": "bbox", "geom": [161, -51, 185, -31]},
    )
    print(d.date(), "DATA" if ds is not None else "no data")

/home/nn411/miniforge3/envs/moana_python/lib/python3.14/site-packages/oceanum/datamesh/connection.py:326: UserWarning: No data found for query
  warnings.warn("No data found for query")


2026-01-01 no data
2026-01-08 no data
2026-01-15 no data
2026-01-22 no data
2026-01-29 no data
2026-02-05 no data
2026-02-12 no data
2026-02-19 no data
2026-02-26 no data
2026-03-05 no data
2026-03-12 no data
2026-03-19 no data
2026-03-26 no data


In [12]:
dsrc = datamesh.get_datasource("era5_temp2m")
print(list(dsrc.variables))
print(dsrc.tstart, "->", dsrc.tend)

['t2m']
1979-01-01 00:00:00+00:00 -> 2026-08-01 00:00:00+00:00


In [15]:
DATASET = "era5_temp2m"

# 1. what the stored data actually covers (lazy, doesn't download it all)
ds = datamesh.load_datasource(DATASET)
print(ds)

# 2. query a few dates across its range
for day in ["2000-01-15", "2020-01-15", "2025-06-15", "2026-01-15", "2026-07-15"]:
    out = datamesh.query(datasource=DATASET, variables=["t2m"],
                         timefilter={"type": "range", "times": [day, day + "T12:00"]},
                         geofilter={"type": "bbox", "geom": [161, -51, 179, -31]})
    print(day, "DATA" if out is not None else "no data")

<xarray.Dataset> Size: 2TB
Dimensions:    (time: 406129, latitude: 721, longitude: 1440)
Coordinates:
  * time       (time) datetime64[ns] 3MB 1979-01-01 ... 2025-05-01
  * latitude   (latitude) float32 3kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float32 6kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
Data variables:
    t2m        (time, latitude, longitude) float32 2TB dask.array<chunksize=(8192, 32, 32), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    cycle:                   2025-04-01T00:00:00
    history:                 2025-05-18T19:23 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts
    _coordinates:            {"t":"time","x":"longitude","y":"latitude"}
    _datamesh_cache_hash:    7f74ee222ea1fae0c15cab9bed8461e3867699ade639c

/home/nn411/miniforge3/envs/moana_python/lib/python3.14/site-packages/oceanum/datamesh/connection.py:326: UserWarning: No data found for query
  warnings.warn("No data found for query")


2025-06-15 no data
2026-01-15 no data
2026-07-15 no data


In [16]:
era5 = ["era5_temp2m", "era5_wind10m", "era5_mslp", "era5_precip_rate",
        "era5_mean_surface_downward_short_wave_radiation_flux"]

for dsid in era5:
    ds = datamesh.load_datasource(dsid)   # lazy, nothing big downloaded
    print(f"{dsid}\n   vars: {list(ds.data_vars)}"
          f"\n   time: {ds.time.min().values} -> {ds.time.max().values}\n")

era5_temp2m
   vars: ['t2m']
   time: 1979-01-01T00:00:00.000000000 -> 2025-05-01T00:00:00.000000000

era5_wind10m
   vars: ['u10', 'v10']
   time: 1940-01-01T00:00:00.000000000 -> 2026-08-01T00:00:00.000000000

era5_mslp
   vars: ['msl']
   time: 1979-01-01T00:00:00.000000000 -> 2026-08-01T00:00:00.000000000

era5_precip_rate
   vars: ['mtpr']
   time: 1979-01-01T00:00:00.000000000 -> 2025-05-01T00:00:00.000000000

era5_mean_surface_downward_short_wave_radiation_flux
   vars: ['msdwswrf']
   time: 2010-01-01T00:00:00.000000000 -> 2020-01-01T00:00:00.000000000

